In [1]:
import pkg_resources
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.core.exceptions import HttpResponseError
import mltable

ml_client = MLClient(
    DefaultAzureCredential()
)

C:\Users\leste\AppData\Local\Temp\ipykernel_10984\1955982348.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:

# glob the parquet file paths for years 2015-19, all months.
paths = [
    {
        "pattern": "wasbs://nyctlc@azureopendatastorage.blob.core.windows.net/green/puYear=2015/puMonth=*/*.parquet"
    },
    {
        "pattern": "wasbs://nyctlc@azureopendatastorage.blob.core.windows.net/green/puYear=2016/puMonth=*/*.parquet"
    },
    {
        "pattern": "wasbs://nyctlc@azureopendatastorage.blob.core.windows.net/green/puYear=2017/puMonth=*/*.parquet"
    },
    {
        "pattern": "wasbs://nyctlc@azureopendatastorage.blob.core.windows.net/green/puYear=2018/puMonth=*/*.parquet"
    },
    {
        "pattern": "wasbs://nyctlc@azureopendatastorage.blob.core.windows.net/green/puYear=2019/puMonth=*/*.parquet"
    },
]

# create a table from the parquet paths
tbl = mltable.from_parquet_files(paths)

# take a random sample
tbl = tbl.take_random_sample(probability=0.001, seed=735)

# filter trips with a distance > 0
tbl = tbl.filter("col('tripDistance') > 0")

# Drop columns
tbl = tbl.drop_columns(["puLocationId", "doLocationId", "storeAndFwdFlag"])

# Create two new columns - year and month - where the values are taken from the path
tbl = tbl.extract_columns_from_partition_format("/puYear={year}/puMonth={month}")

# print the first 5 records of the table as a check
tbl.show(5)

,vendorID,lpepPickupDatetime,lpepDropoffDatetime,passengerCount,tripDistance,pickupLongitude,pickupLatitude,dropoffLongitude,dropoffLatitude,rateCodeID,...,extra,mtaTax,improvementSurcharge,tipAmount,tollsAmount,ehailFee,totalAmount,tripType,year,month
0,2,2015-01-01 21:12:07,2015-01-01 21:15:41,3,0.71,-73.925369,40.761669,-73.923599,40.754658,1,...,0.5,0.5,0.3,1.00,0.0,None,6.80,1,2015,1
1,2,2015-01-01 03:46:04,2015-01-01 04:01:51,1,1.06,-73.964684,40.682896,-73.961937,40.678196,1,...,0.5,0.5,0.3,0.00,0.0,None,7.80,1,2015,1
2,2,2015-01-01 03:55:56,2015-01-01 04:10:34,1,4.90,-73.989822,40.691109,-73.974564,40.646412,1,...,0.5,0.5,0.3,4.38,0.0,None,22.68,1,2015,1
3,2,2015-01-01 03:54:46,2015-01-01 04:14:51,1,3.57,-73.932167,40.707943,-73.963890,40.692127,1,...,0.5,0.5,0.3,0.00,0.0,None,16.80,1,2015,1
4,2,2015-01-01 04:01:51,2015-01-01 04:09:30,5,1.52,-73.889145,40.747181,-73.895729,40.731693,1,...,0.5,0.5,0.3,0.00,0.0,None,8.80,1,2015,1


In [3]:
df=tbl.to_pandas_dataframe()